# PhyVLM-CR: Colab reproduction notebook

Run cells in order. The VLM output supplies a cognitive prior; transmission guides physical inversion and temporal fusion. The notebook queries Sentinel-2 through Earth Engine and calls DashScope Qwen-Image-Edit, so both services require your own access. Set `EE_PROJECT_ID` and `DASHSCOPE_API_KEY` in Colab Secrets before running Cell 0. Do not save a notebook with private credentials or generated outputs before sharing it.


In [ ]:
# @title 0. Colab setup and private credentials
%pip -q install earthengine-api geemap numpy opencv-python-headless matplotlib pillow requests dashscope scikit-image tifffile

import ee
import geemap
from google.colab import userdata

# In Colab > Secrets, add EE_PROJECT_ID and DASHSCOPE_API_KEY.
# Give this notebook access to both secrets. Never paste credentials into cells.
PROJECT_ID = userdata.get("EE_PROJECT_ID")
DASHSCOPE_API_KEY = userdata.get("DASHSCOPE_API_KEY")
if not PROJECT_ID:
    raise ValueError("Add EE_PROJECT_ID to Colab Secrets before running")
if not DASHSCOPE_API_KEY:
    raise ValueError("Add DASHSCOPE_API_KEY to Colab Secrets before running")

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized")


In [ ]:
# @title [Cell 0b] 🛠️ Cell结果打包函数
# --- Canvas stash helper (run once) ---
import io
from PIL import Image
import matplotlib.pyplot as plt

CELL_CANVAS_PNG = {}  # cell_name -> list[png_bytes]

def stash_canvas(cell_name: str, fig=None, dpi=200):
    """Capture current matplotlib figure into PNG bytes (robust in Colab)."""
    global CELL_CANVAS_PNG
    if fig is None:
        fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight")
    CELL_CANVAS_PNG.setdefault(cell_name, []).append(buf.getvalue())


In [ ]:
%%capture cap_cell1 --no-display
# @title [Cell 1] 👉 Real Data Collection
import ee
import geemap
import numpy as np
import matplotlib.pyplot as plt
_cellX_figs_before = set(plt.get_fignums())
# ==========================================
# 1) 参数设置
# ==========================================
TARGET_ID = "20230111T034119_20230111T034634_T48RTN"
TARGET_COORDS = [25.04, 102.70] # [lat, lon]
SCENE_NAME = "Yunnan"

ROI_SIZE_M = 2560
PATCH_SIZE = 512

SEARCH_WINDOW_DAYS = 60
CLEAR_MAX_CLOUD = 20     # 放宽初筛，靠黑边检测把关
MAX_CANDIDATES = 50

# ==========================================
# 2) 下载函数 (保持 Edge Padding 防止误杀)
# ==========================================
def dl(img, region, bands, scale=10):
    arr = geemap.ee_to_numpy(img.select(bands), region=region, scale=scale)
    if arr is None: return None
    if arr.ndim == 2: arr = arr[:, :, np.newaxis]
    arr = np.clip(arr / 3000.0, 0, 1) # Normalize

    # Crop & Pad
    h, w = arr.shape[:2]
    crop = arr[:min(h, PATCH_SIZE), :min(w, PATCH_SIZE)]

    if crop.shape[0] < PATCH_SIZE or crop.shape[1] < PATCH_SIZE:
        crop = np.pad(crop,
            ((0, PATCH_SIZE - crop.shape[0]), (0, PATCH_SIZE - crop.shape[1]), (0, 0)),
            mode="edge" # 关键：用边缘填充，防止黑边误判
        )
    return crop.astype(np.float32)

def dl_prob(img, region):
    arr = geemap.ee_to_numpy(img.select(["MSK_CLDPRB"]), region=region, scale=20)
    if arr is None: return None
    return arr[...,0].astype(np.float32)

# ==========================================
# 3) 核心逻辑：时间戳去重
# ==========================================
def get_triplet_time_dedup(target_id, coords):
    lat, lon = coords
    roi = ee.Geometry.Point([lon, lat]).buffer(ROI_SIZE_M).bounds()

    # --- A. 准备目标影像 ---
    s2_img = ee.Image(f"COPERNICUS/S2_SR_HARMONIZED/{target_id}")
    t1_date = ee.Date(s2_img.get("system:time_start")).format("YYYY-MM-dd").getInfo()
    t1_cloudy = dl(s2_img, roi, ["B4","B3","B2"])
    t1_prob = dl_prob(s2_img, roi)

    if t1_cloudy is None:
        raise RuntimeError("Target Image Download Failed")

    # 获取目标图的 MGRS Tile (减少不必要的跨图幅搜索)
    target_mgrs = s2_img.get("MGRS_TILE").getInfo()

    # --- B. 搜索候选 (Metadata Only) ---
    print(f"1. 搜索 ±{SEARCH_WINDOW_DAYS} 天内的候选 (MGRS: {target_mgrs})...")
    t = ee.Date(s2_img.get("system:time_start"))

    col = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(roi) \
        .filterDate(t.advance(-SEARCH_WINDOW_DAYS, "day"), t.advance(SEARCH_WINDOW_DAYS, "day")) \
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CLEAR_MAX_CLOUD)) \
        .filter(ee.Filter.neq("system:index", target_id)) \
        .filter(ee.Filter.eq("MGRS_TILE", target_mgrs)) \
        .map(lambda im: im.set("time_diff", ee.Number(im.get("system:time_start")).subtract(ee.Number(t.millis())).abs())) \
        .sort("time_diff") \
        .limit(MAX_CANDIDATES)

    candidates_list = col.toList(col.size())
    count = candidates_list.size().getInfo()
    print(f"   -> 找到 {count} 个元数据候选")

    # --- C. 循环筛选 ---
    valid_results = []
    seen_timestamps = set() # ✅ 时间戳去重集合

    for i in range(count):
        if len(valid_results) == 2: break

        ref_ee = ee.Image(candidates_list.get(i))
        ref_id = ref_ee.get("system:index").getInfo()

        # ✅ 关键步骤：获取时间戳并查重
        # GEE 的 system:time_start 是毫秒级，同一拍摄时刻的数据此值完全相同
        ts = ref_ee.get("system:time_start").getInfo()

        if ts in seen_timestamps:
            # print(f"⚠️ 跳过重复时间戳: {ref_id}")
            continue

        # 还没见过这个时间，先下载检查图像质量
        img = dl(ref_ee, roi, ["B4","B3","B2"])

        # 1. 基础检查
        if (img is None) or (not np.isfinite(img).all()):
            continue

        # 2. 黑边检测 (NoData)
        if np.any(np.all(img == 0, axis=-1)):
            continue
        # ==========================================================
        # ✅ 3. ROI 级云量检测 (Nuclear Cloud Check)
        # ==========================================================
        # 下载该候选图的云概率
        cld_prob = dl_prob(ref_ee, roi) # 用低分辨率(60m)快速检查即可
        if cld_prob is None: continue

        # 判定：如果 ROI 内超过 5% 的区域 云概率>40，则视为有云
        cloud_mask = (cld_prob > 40)
        cloud_ratio = np.mean(cloud_mask)

        if cloud_ratio > 0: # 阈值 5%
            print(f"   ⚠️ 跳过局部有云 (ROI Cloud: {cloud_ratio:.1%}): {ref_id}")
            continue

        # ✅ 通过所有检查，记录时间戳，防止选到同一时间的重叠图
        seen_timestamps.add(ts)

        meta = {
            "id": ref_id,
            "date": ee.Date(ref_ee.get("system:time_start")).format("YYYY-MM-dd").getInfo(),
            "diff": float(ref_ee.get("time_diff").getInfo()) / (1000*3600*24)
        }
        valid_results.append((img, meta))
        print(f"   ✅ 选中第 {len(valid_results)} 张: {ref_id} ({meta['date']})")

    # --- D. 结果分发 ---
    if len(valid_results) < 2:
        raise RuntimeError(f"❌ 筛选后仅找到 {len(valid_results)} 张有效图，请扩大搜索范围。")

    real_t2 = valid_results[0][0]
    t2_meta = valid_results[0][1]

    real_t3 = valid_results[1][0]
    t3_meta = valid_results[1][1]

    full_meta = {
        "scene": SCENE_NAME,
        "t1_id": TARGET_ID,
        "t1_date": t1_date,
        "t2_id": t2_meta["id"], "t2_date": t2_meta["date"],
        "t3_id": t3_meta["id"], "t3_date": t3_meta["date"]
    }

    return t1_cloudy, real_t2, real_t3, t1_prob, full_meta

# ==========================================
# 4) 执行与显示
# ==========================================
real_t1_cloudy, real_t2_clear, real_t3_clear, real_t1_prob, meta = get_triplet_time_dedup(TARGET_ID, TARGET_COORDS)

print("\n结果确认:")
print(f"T2 Date: {meta['t2_date']} (ID: {meta['t2_id']})")
print(f"T3 Date: {meta['t3_date']} (ID: {meta['t3_id']})")

# 简单验证一下不是同一张图（基于像素）
print(f"T2/T3 像素是否完全相同? -> {np.array_equal(real_t2_clear, real_t3_clear)}")

def show(arr, title):
    plt.imshow(np.clip(arr,0,1))
    plt.title(title, fontsize=10)
    plt.axis('off')

plt.figure(figsize=(12,3))
plt.subplot(1,4,1); show(real_t1_cloudy, f"T1 Cloudy\n{meta['t1_date']}| 10m")
plt.subplot(1,4,2); show(real_t2_clear, f"T2 Clear\n{meta['t2_date']}| 10m")
plt.subplot(1,4,3); show(real_t3_clear, f"T3 Clear\n{meta['t3_date']}| 10m")
plt.subplot(1,4,4)
if real_t1_prob is not None:
    plt.imshow(real_t1_prob, cmap='inferno', vmin=0, vmax=100)
    plt.title("T1 Cloudy\nMSK_CLDPRB| 20m")
    plt.axis('off')
plt.tight_layout()
stash_canvas("cell1")
plt.show()


_cellX_fignums = [n for n in plt.get_fignums() if n not in _cellX_figs_before]
# 在 cell 末尾（plt.show() 后）加这两行
if "CELL_CANVAS" not in globals(): CELL_CANVAS = {}
CELL_CANVAS.setdefault("cell1", []).append(plt.gcf())

In [ ]:
# @title [Cell 2a] 🧠 Qwen-image-edit 去云
import os
import io
import cv2
import numpy as np
import requests
from PIL import Image
import dashscope
from dashscope import MultiModalConversation

# API key is loaded from Colab Secrets in Cell 0.

# ================= 工具函数 =================
def save_numpy_to_temp_file(img_np, filename):
    """将 float32 numpy 数组保存为本地临时 PNG 文件，返回 file:// 路径"""
    # 转为 0-255 uint8
    img_uint8 = np.clip(img_np * 255, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img_uint8)

    # 保存到 /tmp
    path = f"/tmp/{filename}"
    pil_img.save(path, format='PNG')
    return f"file://{path}"

def download_image_to_numpy(image_url, target_size=None):
    """从 URL 下载结果并转回 numpy"""
    try:
        response = requests.get(image_url, timeout=60)
        response.raise_for_status()
        img_bytes = io.BytesIO(response.content)
        pil_img = Image.open(img_bytes).convert('RGB')

        if target_size is not None:
            pil_img = pil_img.resize(target_size, Image.BICUBIC)

        img_np = np.array(pil_img).astype(np.float32) / 255.0
        return img_np
    except Exception as e:
        print(f"❌ 下载结果失败: {e}")
        return None

# ================= 核心逻辑 =================
def run_qwen_cloud_removal(data):
    print(f"\n🚀 Calling Qwen-Image-Edit-Plus (Multi-Modal)...")

    if not DASHSCOPE_API_KEY:
        print("❌ 错误：请设置 API Key！")
        return None

    # 1. 准备本地临时文件
    # Image 1: 待去云图像 (T1 Simulated)
    path_t1 = save_numpy_to_temp_file(real_t1_cloudy, "real_t1_cloudy.png")

    # 记录原始尺寸
    h, w = real_t1_cloudy.shape[:2]

    # 2. 构建多模态 Prompt
    prompt_text = "去云"#保持无云区地表信息不变，恢复云覆盖区下的地表

    messages = [
        {
            "role": "user",
            "content": [
                {"image": path_t1},
                {"text": prompt_text}
            ]
        }
    ]

    print("  👉 发送多模态请求 (上传图片中)...")

    try:
        # 3. 调用 API
        response = MultiModalConversation.call(
            api_key=DASHSCOPE_API_KEY,
            model="qwen-image-edit-plus", # 使用图像编辑增强版
            messages=messages,
            result_format='message',
            stream=False,
            n=1 # 生成一张
        )

        # 4. 解析结果
        if response.status_code == 200:
            print("  ✅ API 调用成功！")

            # 解析返回的 content
            # response 结构通常是: output -> choices -> message -> content -> image
            try:
                content_list = response.output.choices[0].message.content
                result_url = None
                for item in content_list:
                    if 'image' in item:
                        result_url = item['image']
                        break

                if result_url:
                    print("  ⬇️ 正在下载结果...")
                    rec_img = download_image_to_numpy(result_url, target_size=(w, h))
                    return rec_img
                else:
                    print("❌ 返回结果中未找到图片 URL")
                    print(response)
                    return None
            except Exception as e:
                print(f"❌ 解析响应失败: {e}")
                print(response)
                return None
        else:
            print(f"❌ API 报错: {response.code}, {response.message}")
            return None

    except Exception as e:
        print(f"❌ 发生异常: {e}")
        return None

# --- 执行入口 ---
if __name__ == "__main__":
    if real_t1_cloudy is not None:
        Qwen_result= run_qwen_cloud_removal(real_t1_cloudy)
        if Qwen_result is not None:
            print("✅ Qwen重建完成！")
        else:
            print("❌ 重建失败。")
    else:
        print("⚠️ 请先运行 [Cell 1] 生成 input_data 数据！")

In [ ]:
%%capture cap_cell2 --no-display
# @title [Cell 2b] 📈 Qwen 重建结果评估与展示
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import matplotlib.pyplot as plt
import numpy as np
_cellX_figs_before = set(plt.get_fignums())

def vis_reconstruction_result(real_t1_cloudy, Qwen_result, real_t2_clear):
    """
    可视化Qwen去云结果 (6图布局)
    展示: Cloudy, Qwen_result, Clear
        Error1        Error2
    """

    # 计算指标
    gt = np.clip(real_t2_clear, 0, 1).astype(np.float32)
    res = np.clip(Qwen_result, 0, 1).astype(np.float32)
    # PSNR
    val_psnr = psnr(gt, res, data_range=1.0)
    # SSIM win_size=7 是 SSIM 的标准稳健设置
    val_ssim = ssim(gt, res, data_range=1.0, channel_axis=2, win_size=7)

    print(f"\n📈 Qwen Evaluation: PSNR: {val_psnr:.2f} dB | SSIM: {val_ssim:.4f}")

    # 绘图
    plt.figure(figsize=(9, 6))

    def show(idx, img, title, cmap=None):
        plt.subplot(2, 3, idx)
        if img.ndim == 3 and img.shape[2] == 1: img = img[:,:,0]
        plt.imshow(np.clip(img, 0, 1), cmap=cmap)
        plt.title(title, fontsize=11)
        plt.axis('off')




    # Row 1: 影像
    show(1, real_t1_cloudy, "T1 Cloudy")
    show(2, Qwen_result, f"Qwen Result\nPSNR: {val_psnr:.2f} | SSIM: {val_ssim:.4f}")
    show(3, real_t2_clear, "T2 Clear")

    # Row 2: Diff (放大5倍以便观察)
    diff = np.abs(real_t1_cloudy - Qwen_result) * 5
    show(4, diff, "Diff (Cloudy/Result)", cmap='inferno')

    plt.subplot(2,3,5).axis("off") # 空白不展示任何内容

    diff = np.abs(real_t2_clear - Qwen_result) * 5
    show(6, diff, "Diff (Clear/Result)", cmap='inferno')

    plt.tight_layout()
    stash_canvas("cell2")
    plt.show()

# --- 执行入口 ---
if __name__ == "__main__":
    if real_t1_cloudy is not None:
        if Qwen_result is not None:
            vis_reconstruction_result(real_t1_cloudy, Qwen_result, real_t2_clear)
        else:
            print("❌ 重建结果为空。")
    else:
        print("⚠️ 请确保先运行了 [Cell 1] 和 [Cell 2a]！")

_cellX_fignums = [n for n in plt.get_fignums() if n not in _cellX_figs_before]
# 在 cell 末尾（plt.show() 后）加这两行
if "CELL_CANVAS" not in globals(): CELL_CANVAS = {}
CELL_CANVAS.setdefault("cell2", []).append(plt.gcf())


In [ ]:
%%capture cap_cell3 --no-display
# @title [Cell 3] ✅ VLM认知先验的物理转译：A, T

import numpy as np
import cv2
import matplotlib.pyplot as plt

_cellX_figs_before = set(plt.get_fignums())


# ============================================================
# Basic utilities
# ============================================================

def to_float01(x):
    """Convert image to float32 RGB image in [0, 1]."""
    x = np.asarray(x)
    if x.dtype == np.uint8:
        x = x.astype(np.float32) / 255.0
    else:
        x = x.astype(np.float32)
        if np.nanmax(x) > 1.5:
            x = np.clip(x / 255.0, 0, 1)
        else:
            x = np.clip(x, 0, 1)
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)
    if x.shape[-1] != 3:
        raise ValueError(f"Expect image shape (H,W,3), got {x.shape}")
    return x.astype(np.float32)


def align_to(src, ref):
    """Resize src to the spatial size of ref."""
    if src.shape[:2] == ref.shape[:2]:
        return src
    h, w = ref.shape[:2]
    return cv2.resize(src, (w, h), interpolation=cv2.INTER_LINEAR)


def rgb_to_hsv01(rgb01):
    """Convert RGB image in [0,1] to HSV image in [0,1]."""
    hsv = cv2.cvtColor((np.clip(rgb01, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 0] /= 179.0
    hsv[..., 1] /= 255.0
    hsv[..., 2] /= 255.0
    return hsv


def grad_mag(gray01):
    """Gradient magnitude of a gray image."""
    g = gray01.astype(np.float32)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    return np.sqrt(gx * gx + gy * gy + 1e-8)


def robust_norm01(x, lo=5, hi=95):
    """Robust percentile normalization to [0,1]."""
    a = np.nanpercentile(x, lo)
    b = np.nanpercentile(x, hi)
    return np.clip((x - a) / (b - a + 1e-6), 0, 1).astype(np.float32)


# ============================================================
# Cloud-veil candidate selection
# ============================================================

def whiteness_gate(I_rgb01, v0=0.60, s0=0.25, alpha=10.0, beta=10.0):
    """Cloud-like whiteness likelihood: high brightness + low saturation."""
    hsv = rgb_to_hsv01(I_rgb01)
    s = hsv[..., 1]
    v = hsv[..., 2]
    g_v = 1.0 / (1.0 + np.exp(-alpha * (v - v0)))
    g_s = 1.0 / (1.0 + np.exp(-beta * (s0 - s)))
    return np.clip(g_v * g_s, 0, 1).astype(np.float32)


def low_texture_gate(I_gray01, win=21, t=0.35, k=10.0):
    """Low-texture likelihood for cloud-veil candidate selection."""
    G = grad_mag(I_gray01)
    local_texture = cv2.blur(G, (win, win))
    texture01 = robust_norm01(local_texture)
    gate = 1.0 / (1.0 + np.exp(-k * (t - texture01)))
    return np.clip(gate, 0, 1).astype(np.float32), texture01


# ============================================================
# Radiometric normalization of VLM prior
# ============================================================

def radiometric_align_by_clear_pixels(I_ref, V_src, clear_mask, eps=1e-6):
    """Align VLM prior to the observation using robust percentile matching over relatively clear pixels."""
    V_adj = V_src.copy()
    if clear_mask is None or clear_mask.sum() < 500:
        a = np.ones(3, np.float32)
        b = np.zeros(3, np.float32)
        return V_adj.astype(np.float32), (a, b)

    a = np.ones(3, np.float32)
    b = np.zeros(3, np.float32)
    for c in range(3):
        x = V_src[..., c][clear_mask]
        y = I_ref[..., c][clear_mask]
        x1, x2 = np.percentile(x, [10, 90])
        y1, y2 = np.percentile(y, [10, 90])
        aa = (y2 - y1) / (x2 - x1 + eps)
        bb = y1 - aa * x1
        a[c], b[c] = aa, bb
        V_adj[..., c] = np.clip(aa * V_src[..., c] + bb, 0, 1)
    return V_adj.astype(np.float32), (a, b)


# ============================================================
# Base-layer operator
# ============================================================

def base_layer(img01, sigma_color=0.10, sigma_space=7):
    """
    Edge-preserving base layer.
    Used to fit A and T while suppressing local VLM textures.
    """
    img01 = np.clip(img01.astype(np.float32), 0, 1)
    base = []
    for c in range(3):
        base_c = cv2.bilateralFilter(
            img01[..., c],
            d=0,
            sigmaColor=float(sigma_color),
            sigmaSpace=int(sigma_space)
        )
        base.append(base_c)
    return np.stack(base, axis=-1).astype(np.float32)


# ============================================================
# Main function: VLM prior -> A, T
# ============================================================

def physics_estimate_AIprior(
    Ic,
    Iq,
    win_tex=21,
    cloud_top_q=85,
    clear_low_q=20,
    base_sigma_color=0.10,
    base_sigma_space=7,
    t_min=0.02,
    show=True,
):
    """
    Translate the VLM cognitive prior into the two physical variables used by the simplified PhyVLM-CR6 pipeline.

    Inputs
    ------
    Ic : cloudy observation I
    Iq : VLM cloud-free prior J_VLM

    Outputs
    -------
    T       : transmission / physical visibility map
    A_rgb   : global atmospheric light
    Iq_adj  : radiometrically normalized VLM prior
    W_cloud : cloud-veil candidate likelihood
    Ic_hat_b: base-layer cloudy image re-synthesized from J_VLM, A, and T
    """

    # ------------------------------------------------------------
    # 1. Standardize inputs
    # ------------------------------------------------------------
    Ic = to_float01(Ic)
    Iq = align_to(to_float01(Iq), Ic)

    IcGray = cv2.cvtColor((Ic * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0

    # ------------------------------------------------------------
    # 2. Cloud-veil candidate selection for atmospheric light
    # ------------------------------------------------------------
    W_white = whiteness_gate(Ic)
    W_lowtex, texture01 = low_texture_gate(IcGray, win=win_tex)
    W_cloud = np.clip(W_white * W_lowtex, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 3. Radiometric normalization of VLM prior
    # ------------------------------------------------------------
    thr_clear = np.percentile(W_cloud, clear_low_q)
    clear_mask = (W_cloud <= thr_clear)
    Iq_adj, (align_a, align_b) = radiometric_align_by_clear_pixels(
        I_ref=Ic,
        V_src=Iq,
        clear_mask=clear_mask
    )

    # ------------------------------------------------------------
    # 4. Atmospheric light estimation
    # ------------------------------------------------------------
    thr_cloud = np.percentile(W_cloud, cloud_top_q)
    cloud_mask = (W_cloud >= thr_cloud)
    if cloud_mask.sum() < 500:
        v = rgb_to_hsv01(Ic)[..., 2]
        cloud_mask = (v >= np.percentile(v, 95))

    A_rgb = np.median(Ic[cloud_mask], axis=0).astype(np.float32)
    A_rgb = np.clip(A_rgb, 0, 1)
    A_img = np.ones_like(Ic, dtype=np.float32) * A_rgb[None, None, :]

    # ------------------------------------------------------------
    # 5. Base-layer fitting of transmission
    # ------------------------------------------------------------
    Ic_b = base_layer(Ic, sigma_color=base_sigma_color, sigma_space=base_sigma_space)
    Iq_b = base_layer(Iq_adj, sigma_color=base_sigma_color, sigma_space=base_sigma_space)

    # From I - A ≈ t * (J_VLM - A), obtain a per-pixel least-squares estimate of t.
    P = (Ic_b - A_img).astype(np.float32)
    Q = (Iq_b - A_img).astype(np.float32)
    num = np.sum(P * Q, axis=-1)
    den = np.sum(Q * Q, axis=-1) + 1e-6
    T = np.clip(num / den, float(t_min), 1.0).astype(np.float32)

    # Re-synthesized cloudy observation used only for visualization/diagnostics.
    Ic_hat_b = T[..., None] * Iq_b + (1.0 - T)[..., None] * A_img

    # ------------------------------------------------------------
    # 6. Visualization
    # ------------------------------------------------------------
    if show:
        plt.figure(figsize=(9, 9))

        def _im(ax, img, title, cmap=None, vmin=None, vmax=None):
            ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(title, fontsize=11)
            ax.axis("off")

        ax = plt.subplot(3, 3, 1)
        _im(ax, Ic, "cloudy observation I")

        ax = plt.subplot(3, 3, 2)
        _im(ax, Iq, "VLM prior J_VLM")

        ax = plt.subplot(3, 3, 3)
        _im(ax, Iq_adj, f"normalized VLM prior\na={align_a.round(3)} b={align_b.round(3)}")

        ax = plt.subplot(3, 3, 4)
        _im(ax, W_cloud, "cloud-veil candidate", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(3, 3, 5)
        _im(ax, T, "transmission T", cmap="gray", vmin=0, vmax=1)

        A_patch = np.ones((80, 80, 3), np.float32) * A_rgb[None, None, :]
        ax = plt.subplot(3, 3, 6)
        _im(ax, A_patch, "atmospheric light A")

        ax = plt.subplot(3, 3, 7)
        _im(ax, Ic_b, "base cloudy I^B")

        ax = plt.subplot(3, 3, 8)
        _im(ax, Iq_b, "base VLM prior J^B_VLM")

        ax = plt.subplot(3, 3, 9)
        residual = np.linalg.norm(Ic_b - Ic_hat_b, axis=-1)
        _im(ax, residual, "diagnostic re-synthesis residual", cmap="gray")

        plt.tight_layout()
        stash_canvas("cell3")
        plt.show()

        print(
            f"[Cell3 Stats] T mean={T.mean():.3f} | "
            f"T min/max=({T.min():.3f}, {T.max():.3f}) | "
            f"A={A_rgb.round(4)}"
        )

    return T, A_rgb, Iq_adj, W_cloud, Ic_hat_b


# ============================================================
# Run Cell 3
# ============================================================

T, A_rgb, Iq_adj, W_cloud, Ic_hat = physics_estimate_AIprior(
    real_t1_cloudy,
    Qwen_result,
    show=True
)

_cellX_fignums = [n for n in plt.get_fignums() if n not in _cellX_figs_before]

if "CELL_CANVAS" not in globals():
    CELL_CANVAS = {}

CELL_CANVAS.setdefault("cell3", []).append(plt.gcf())


In [ ]:
%%capture cap_cell4 --no-display
# @title [Cell 4] ✅ CR6: 以 transmission 为核心的物理-认知-时相联合去云

import numpy as np
import cv2
import matplotlib.pyplot as plt

_cellX_figs_before = set(plt.get_fignums())


# ============================================================
# Basic utilities
# ============================================================

def to_float01_rgb(x):
    """Convert image to float32 RGB image in [0,1]."""
    x = np.asarray(x)
    if x.dtype == np.uint8:
        x = x.astype(np.float32) / 255.0
    else:
        x = x.astype(np.float32)
        if np.nanmax(x) > 1.5:
            x = np.clip(x / 255.0, 0, 1)
        else:
            x = np.clip(x, 0, 1)
    if x.ndim == 2:
        x = np.stack([x, x, x], axis=-1)
    if x.shape[-1] != 3:
        raise ValueError(f"Expect image shape (H,W,3), got {x.shape}")
    return x.astype(np.float32)


def align_to(src, ref):
    """Resize src to the spatial size of ref."""
    if src.shape[:2] == ref.shape[:2]:
        return src
    h, w = ref.shape[:2]
    return cv2.resize(src, (w, h), interpolation=cv2.INTER_LINEAR)


def water_likelihood_from_rgb(rgb01):
    """
    RGB-only water likelihood used for low-texture water regularization.
    It is estimated from the aligned temporal reference, not from the cloudy input.
    """
    rgb01 = np.clip(rgb01.astype(np.float32), 0, 1)
    hsv = cv2.cvtColor((rgb01 * 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 0] /= 179.0
    hsv[..., 1] /= 255.0
    hsv[..., 2] /= 255.0

    gray = cv2.cvtColor((rgb01 * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    texture = cv2.blur(np.abs(cv2.Laplacian(gray, cv2.CV_32F)), (21, 21))
    lo, hi = np.percentile(texture, [5, 95])
    texture01 = np.clip((texture - lo) / (hi - lo + 1e-6), 0, 1)

    dark = 1.0 / (1.0 + np.exp(12.0 * (hsv[..., 2] - 0.48)))
    low_texture = 1.0 / (1.0 + np.exp(12.0 * (texture01 - 0.32)))
    saturated = 1.0 / (1.0 + np.exp(-12.0 * (hsv[..., 1] - 0.08)))

    return np.clip(dark * low_texture * saturated, 0, 1).astype(np.float32)


def per_channel_affine_align(src, ref, mask, eps=1e-6):
    """Radiometrically align temporal reference to current observation over reliable regions."""
    src = np.clip(src.astype(np.float32), 0, 1)
    ref = np.clip(ref.astype(np.float32), 0, 1)
    out = src.copy()
    a = np.ones(3, np.float32)
    b = np.zeros(3, np.float32)

    if mask is None or mask.sum() < 500:
        return out.astype(np.float32), a, b

    for c in range(3):
        x = src[..., c][mask]
        y = ref[..., c][mask]
        x1, x2 = np.percentile(x, [10, 90])
        y1, y2 = np.percentile(y, [10, 90])
        aa = (y2 - y1) / (x2 - x1 + eps)
        bb = y1 - aa * x1
        a[c], b[c] = aa, bb
        out[..., c] = np.clip(aa * src[..., c] + bb, 0, 1)
    return out.astype(np.float32), a, b


def low_pass(img01, sigma):
    """Gaussian low-pass filtering."""
    return cv2.GaussianBlur(
        np.clip(img01.astype(np.float32), 0, 1),
        (0, 0),
        sigmaX=float(sigma)
    ).astype(np.float32)


def detail_layer(img01, sigma, clip_value=0.18):
    """Observation detail layer. High-frequency details are inherited from I, not from the VLM prior."""
    base = low_pass(img01, sigma=sigma)
    detail = img01.astype(np.float32) - base
    return np.clip(detail, -clip_value, clip_value).astype(np.float32)


# ============================================================
# Main function: simplified transmission-centered restoration
# ============================================================

def radiometric_decloud_sharper(
    IcRGB,
    IqRGB,
    TrefRGB,
    T,
    A_rgb,
    k_vis=4.0,
    lp_sigma=8.0,
    guide_strength=0.8,
    detail_sigma=2.0,
    detail_strength=1.0,
    detail_clip=0.18,
    refine_T=True,
    refine_sigma=1.2,
    t0=0.05,
    preserve_clear=False,
    clear_thr=0.97,
    water_regularize=True,
    water_strength=0.65,
    water_sigma=2.0,
    show=True,
):
    """
    CR6 simplified logic:
    (I, J_VLM) -> (A, T) -> J_phy -> J_cog -> J.

    T is the central physical reliability variable:
    1. T controls physical inversion stability.
    2. T controls low-frequency VLM cognitive adjustment.
    3. T controls observed detail preservation.
    4. exp[-k(1-T)] controls temporal-reference fusion.
    """

    # ------------------------------------------------------------
    # 1. Standardize and align inputs
    # ------------------------------------------------------------
    Ic = to_float01_rgb(IcRGB)
    V = align_to(to_float01_rgb(IqRGB), Ic)
    Iref = align_to(to_float01_rgb(TrefRGB), Ic)

    # ------------------------------------------------------------
    # 2. Prepare transmission map
    # ------------------------------------------------------------
    T = np.asarray(T).astype(np.float32)
    if T.ndim == 3:
        T = T[..., 0]
    T = align_to(T, Ic)
    T = np.clip(T, 0, 1)

    if refine_T:
        T = cv2.GaussianBlur(T, (0, 0), sigmaX=float(refine_sigma))
        T = np.clip(T, 0, 1).astype(np.float32)

    H = (1.0 - T).astype(np.float32)

    # Visibility-controlled final fusion weight.
    W_vis = np.exp(-float(k_vis) * H).astype(np.float32)
    W_ref = 1.0 - W_vis

    # ------------------------------------------------------------
    # 3. Physical radiometric correction
    # ------------------------------------------------------------
    A_rgb = np.clip(np.asarray(A_rgb).astype(np.float32), 0, 1)
    A_img = np.ones_like(Ic, dtype=np.float32) * A_rgb[None, None, :]

    T_safe = np.maximum(T, float(t0)).astype(np.float32)
    J_phy = (Ic - H[..., None] * A_img) / T_safe[..., None]
    J_phy = np.clip(J_phy, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 4. Transmission-guided low-frequency VLM cognitive adjustment
    # ------------------------------------------------------------
    J_phy_lp = low_pass(J_phy, sigma=lp_sigma)
    V_lp = low_pass(V, sigma=lp_sigma)
    Delta_vlm_lp = (V_lp - J_phy_lp).astype(np.float32)

    # VLM prior is admitted only where the current observation is physically recoverable.
    W_cog = T.astype(np.float32)
    J_cog_lf = J_phy + guide_strength * W_cog[..., None] * Delta_vlm_lp
    J_cog_lf = np.clip(J_cog_lf, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 5. Transmission-guided observation detail preservation
    # ------------------------------------------------------------
    D_obs = detail_layer(Ic, sigma=detail_sigma, clip_value=detail_clip)
    W_detail = T.astype(np.float32)
    J_cog = J_cog_lf + detail_strength * W_detail[..., None] * D_obs
    J_cog = np.clip(J_cog, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 6. Temporal reference alignment
    # ------------------------------------------------------------
    ref_align_mask = (T > 0.3)
    J_ref, ref_a, ref_b = per_channel_affine_align(
        src=Iref,
        ref=Ic,
        mask=ref_align_mask
    )

    # ------------------------------------------------------------
    # 7. Visibility-controlled final fusion
    # ------------------------------------------------------------
    J = W_vis[..., None] * J_cog + W_ref[..., None] * J_ref
    J = np.clip(J, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 8. Optional low-texture water regularization
    # ------------------------------------------------------------
    W_water = np.zeros_like(T, dtype=np.float32)
    if water_regularize:
        W_water = water_likelihood_from_rgb(J_ref)
        J_ref_water = cv2.GaussianBlur(J_ref, (0, 0), sigmaX=float(water_sigma))
        Ww = np.clip(float(water_strength) * W_water, 0, 1).astype(np.float32)
        J = np.clip((1.0 - Ww[..., None]) * J + Ww[..., None] * J_ref_water, 0, 1).astype(np.float32)

    # ------------------------------------------------------------
    # 9. Optional clear-pixel preservation
    # ------------------------------------------------------------
    if preserve_clear:
        clear_keep = (T > clear_thr)
        J[clear_keep] = Ic[clear_keep]

    # ------------------------------------------------------------
    # 10. Visualization
    # ------------------------------------------------------------
    if show:
        plt.figure(figsize=(7, 18))

        def _im(ax, img, title, cmap=None, vmin=None, vmax=None):
            ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(title, fontsize=11)
            ax.axis("off")

        ax = plt.subplot(6, 2, 1)
        _im(ax, Ic, "cloudy observation I")

        ax = plt.subplot(6, 2, 2)
        _im(ax, V, "VLM prior J_VLM")

        ax = plt.subplot(6, 2, 3)
        _im(ax, T, "transmission T", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(6, 2, 4)
        _im(ax, W_vis, "visibility weight ω", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(6, 2, 5)
        _im(ax, J_phy, "physical inversion J_phy")

        Delta_norm = np.linalg.norm(Delta_vlm_lp, axis=-1)
        Delta_norm = np.clip(
            (Delta_norm - np.percentile(Delta_norm, 5)) /
            (np.percentile(Delta_norm, 95) - np.percentile(Delta_norm, 5) + 1e-6),
            0, 1
        )
        ax = plt.subplot(6, 2, 6)
        _im(ax, Delta_norm, "low-frequency VLM correction", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(6, 2, 7)
        _im(ax, W_cog, "cognitive weight T", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(6, 2, 8)
        _im(ax, J_cog_lf, "low-frequency adjusted result")

        D_norm = np.linalg.norm(D_obs, axis=-1)
        D_norm = np.clip(
            (D_norm - np.percentile(D_norm, 5)) /
            (np.percentile(D_norm, 95) - np.percentile(D_norm, 5) + 1e-6),
            0, 1
        )
        ax = plt.subplot(6, 2, 9)
        _im(ax, D_norm, "observation detail D(I)", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(6, 2, 10)
        _im(ax, J_cog, "cognitively adjusted J_cog")

        ax = plt.subplot(6, 2, 11)
        _im(ax, W_ref, "reference weight 1-ω", cmap="gray", vmin=0, vmax=1)

        ax = plt.subplot(6, 2, 12)
        _im(ax, J, "final restoration J")

        plt.tight_layout()
        stash_canvas("cell4")
        plt.show()

        print(
            f"[Cell4 Stats] mean T={T.mean():.3f} | "
            f"mean W_vis={W_vis.mean():.3f} | "
            f"mean W_ref={W_ref.mean():.3f} | "
            f"ref_a={ref_a.round(3)} ref_b={ref_b.round(3)}"
        )

    # Save intermediate results for ablation and visualization.
    globals()["CELL4_DEBUG"] = dict(
        T=T,
        W_vis=W_vis,
        W_ref=W_ref,
        W_cog=W_cog,
        W_detail=W_detail,
        J_phy=J_phy,
        Delta_vlm_lp=Delta_vlm_lp,
        J_cog_lf=J_cog_lf,
        D_obs=D_obs,
        J_cog=J_cog,
        W_water=W_water,
        J_ref=J_ref,
        J=J,
    )

    return J


# ============================================================
# Run Cell 4
# ============================================================

out = radiometric_decloud_sharper(
    real_t1_cloudy,
    Qwen_result,
    real_t3_clear,
    T,
    A_rgb,
    k_vis=4.0,
    lp_sigma=8.0,
    guide_strength=0.8,
    detail_sigma=2.0,
    detail_strength=1.0,
    detail_clip=0.18,
    refine_T=True,
    refine_sigma=1.2,
    t0=0.05,
    preserve_clear=False,
    clear_thr=0.97,
    water_regularize=True,
    water_strength=0.65,
    water_sigma=2.0,
    show=True
)

S_decloud = out

_cellX_fignums = [n for n in plt.get_fignums() if n not in _cellX_figs_before]

if "CELL_CANVAS" not in globals():
    CELL_CANVAS = {}

CELL_CANVAS.setdefault("cell4", []).append(plt.gcf())


In [ ]:
%%capture cap_cell5 --no-display
# @title [Cell 5] ✅ Quantitative Eval: PSNR/SSIM/ErrorMaps(MAE)
import numpy as np
import cv2
import matplotlib.pyplot as plt
import math
_cellX_figs_before = set(plt.get_fignums())

# ---- helpers ----
def _to_float01_rgb(x):
    x = np.asarray(x)
    if x.dtype == np.uint8:
        x = x.astype(np.float32)/255.0
    else:
        x = x.astype(np.float32)
        if np.nanmax(x) > 1.5:
            x = np.clip(x/255.0, 0, 1)
        else:
            x = np.clip(x, 0, 1)
    if x.ndim == 2:
        x = np.stack([x,x,x], -1)
    if x.shape[-1] != 3:
        raise ValueError(f"Expect (H,W,3), got {x.shape}")
    return np.clip(x, 0, 1).astype(np.float32)

def _align_to(src, ref):
    if src.shape[:2] == ref.shape[:2]:
        return src
    h, w = ref.shape[:2]
    return cv2.resize(src, (w, h), interpolation=cv2.INTER_LINEAR)

def psnr01(pred01, gt01, eps=1e-12):
    pred = np.clip(pred01, 0, 1).astype(np.float32)
    gt   = np.clip(gt01,   0, 1).astype(np.float32)
    mse = float(np.mean((pred - gt) ** 2))
    if mse < eps:
        return float("inf")
    return 20.0 * math.log10(1.0 / math.sqrt(mse))

# try skimage SSIM; fallback to a simple implementation if not available
def ssim_rgb(pred01, gt01):
    try:
        from skimage.metrics import structural_similarity as ssim
        # channel_axis for new skimage; if old, use multichannel=True
        try:
            return float(ssim(gt01, pred01, data_range=1.0, channel_axis=-1))
        except TypeError:
            return float(ssim(gt01, pred01, data_range=1.0, multichannel=True))
    except Exception as e:
        print("⚠️ skimage not available, SSIM fallback (approx). Error:", e)
        # Very rough fallback: average SSIM over channels using Gaussian window
        def _ssim_1ch(x, y):
            x = x.astype(np.float32); y = y.astype(np.float32)
            K1, K2 = 0.01, 0.03
            L = 1.0
            C1, C2 = (K1*L)**2, (K2*L)**2
            mu_x = cv2.GaussianBlur(x, (11,11), 1.5)
            mu_y = cv2.GaussianBlur(y, (11,11), 1.5)
            mu_x2 = mu_x*mu_x
            mu_y2 = mu_y*mu_y
            mu_xy = mu_x*mu_y
            sig_x2 = cv2.GaussianBlur(x*x, (11,11), 1.5) - mu_x2
            sig_y2 = cv2.GaussianBlur(y*y, (11,11), 1.5) - mu_y2
            sig_xy = cv2.GaussianBlur(x*y, (11,11), 1.5) - mu_xy
            ssim_map = ((2*mu_xy + C1)*(2*sig_xy + C2))/((mu_x2 + mu_y2 + C1)*(sig_x2 + sig_y2 + C2) + 1e-12)
            return float(np.mean(ssim_map))
        return float(np.mean([_ssim_1ch(pred01[...,c], gt01[...,c]) for c in range(3)]))

def error_map_mae(pred01, gt01):
    """per-pixel mean abs error over RGB, in [0,1]"""
    return np.mean(np.abs(pred01 - gt01), axis=-1).astype(np.float32)

def error_map_rmse(pred01, gt01):
    """per-pixel RMSE over RGB, in [0,1]"""
    return np.sqrt(np.mean((pred01 - gt01)**2, axis=-1) + 1e-12).astype(np.float32)

# ---- pick variables robustly ----
if "S_fused" in globals() and S_fused is not None:
    pred = S_fused
elif "S_decloud" in globals() and S_decloud is not None:
    pred = S_decloud
elif "out" in globals() and isinstance(out, dict) and "S" in out:
    pred = out["S"]
else:
    raise RuntimeError("找不到 s_fused 输出。请确认变量名为 S_fused / S_decloud / out['S']。")

if "real_t2_clear" not in globals() or real_t2_clear is None:
    raise RuntimeError("找不到 real_t2_clear。")
if "real_t1_cloudy" not in globals() or real_t1_cloudy is None:
    raise RuntimeError("找不到 real_t1_cloudy。")

pred01 = _to_float01_rgb(pred)
gt01   = _to_float01_rgb(real_t2_clear)
ic01   = _to_float01_rgb(real_t1_cloudy)
Qwen01 = _to_float01_rgb(Qwen_result)

pred01 = _align_to(pred01, gt01)
ic01   = _align_to(ic01,   gt01)

# ---- metrics ----
psnr_val = psnr01(pred01, gt01)
ssim_val = ssim_rgb(pred01, gt01)

print(f"✅ PSNR(CoPhyCR vs real_t2_clear) = {psnr_val:.3f} dB")
print(f"✅ SSIM(CoPhyCR vs real_t2_clear) = {ssim_val:.4f}")

psnr_val_Qwen = psnr01(Qwen01, gt01)
ssim_val_Qwen = ssim_rgb(Qwen01, gt01)

print(f"✅ PSNR(Qwen vs real_t2_clear) = {psnr_val_Qwen:.3f} dB")
print(f"✅ SSIM(Qwen vs real_t2_clear) = {ssim_val_Qwen:.4f}")
# ---- error maps ----
err_vs_clear_mae = error_map_mae(pred01, gt01)
err_vs_clear_mae_Qwen = error_map_mae(Qwen01, gt01)

# visualization scaling for error maps
# vmax_mae = float(np.percentile(np.concatenate([err_vs_clear_mae.reshape(-1), err_vs_clear_mae_Qwen.reshape(-1)]), 99))

plt.figure(figsize=(9,6))

# row 1: images
ax = plt.subplot(2,3,1); ax.imshow(ic01); ax.set_title("real_t1_cloudy"); ax.axis("off")
ax = plt.subplot(2,3,2); ax.imshow(pred01); ax.set_title("AllcloudRemoval"); ax.axis("off")
ax = plt.subplot(2,3,3); ax.imshow(gt01); ax.set_title("real_t2_clear"); ax.axis("off")

# row 2: error maps (MAE)
ax = plt.subplot(2,3,4);
im = ax.imshow(err_vs_clear_mae, cmap="magma", vmin=0, vmax=1)
ax.set_title(f"Error_map_CoPhyCR"); ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.subplot(2,3,5).axis("off") # 空白不展示任何内容

ax = plt.subplot(2,3,6);
im = ax.imshow(err_vs_clear_mae_Qwen, cmap="magma", vmin=0, vmax=1)
ax.set_title(f"Error_map_Qwen"); ax.axis("off")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
stash_canvas("cell5")
plt.show()

print("[Error Stats] CoPhyCR vs GT: mean={:.4f}, p95={:.4f}".format(float(err_vs_clear_mae.mean()), float(np.percentile(err_vs_clear_mae,95))))
print("[Error Stats] Qwen vs GT:  mean={:.4f}, p95={:.4f}".format(float(err_vs_clear_mae_Qwen.mean()), float(np.percentile(err_vs_clear_mae_Qwen,95))))
_cellX_fignums = [n for n in plt.get_fignums() if n not in _cellX_figs_before]
# 在 cell 末尾（plt.show() 后）加这两行
if "CELL_CANVAS" not in globals(): CELL_CANVAS = {}
CELL_CANVAS.setdefault("cell5", []).append(plt.gcf())


In [ ]:
# @title [Cell 6] 打包下载
import os, re, json, time, zipfile, sys
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt

# --- TIFF backend ---
try:
    import tifffile
    _HAS_TIFF = True
except Exception:
    _HAS_TIFF = False

def _safe(s: str) -> str:
    s = str(s)
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9a-zA-Z_\-\.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s[:160]

def _coords_str(coords):
    arr = np.asarray(coords).reshape(-1)
    parts = []
    for v in arr:
        try:
            parts.append(f"{float(v):.6f}".rstrip("0").rstrip("."))
        except Exception:
            parts.append(str(v))
    return "x".join(parts)

def save_tif(path, arr):
    a = np.asarray(arr)
    if a.dtype in (np.float16, np.float64):
        a = a.astype(np.float32)
    if a.ndim == 1 and a.shape[0] == 3:
        a = a.reshape(1,1,3)
    if _HAS_TIFF:
        tifffile.imwrite(str(path), a)
    else:
        np.save(str(path) + ".npy", a)

def save_jpg_rgb01(path, rgb01, quality=95):
    x = np.asarray(rgb01).astype(np.float32)
    if np.nanmax(x) > 1.5:
        x = np.clip(x/255.0, 0, 1)
    else:
        x = np.clip(x, 0, 1)
    u8 = (x*255.0 + 0.5).astype(np.uint8)  # RGB
    bgr = cv2.cvtColor(u8, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])

def save_jpg_map_scaled(path, map2d, vmin, vmax, cmap="magma", quality=95):
    m = np.asarray(map2d).astype(np.float32)
    if m.ndim == 3:
        m = m[...,0]
    x01 = np.clip((m - vmin) / (vmax - vmin + 1e-6), 0, 1)
    u8 = (x01*255.0 + 0.5).astype(np.uint8)

    cmap = cmap.lower()
    if cmap == "magma":
        cm = cv2.applyColorMap(u8, cv2.COLORMAP_MAGMA)
    elif cmap == "inferno":
        cm = cv2.applyColorMap(u8, cv2.COLORMAP_INFERNO)
    else:
        cm = cv2.applyColorMap(u8, cv2.COLORMAP_TURBO)

    cv2.imwrite(str(path), cm, [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])

def save_patch_from_rgb(path, rgb_vec, size=80):
    c = np.asarray(rgb_vec).astype(np.float32).reshape(1,1,3)
    patch = np.ones((size,size,3), np.float32) * c
    save_jpg_rgb01(path, patch)

def get_cap_stdout(cap_name):
    cap = globals().get(cap_name, None)
    if cap is None:
        return None
    return getattr(cap, "stdout", None)

def save_print_txt(cell_name, cap_name, process_dir):
    txt = get_cap_stdout(cap_name)
    if txt is None:
        txt = (f"(No captured stdout for {cell_name}. "
               f"Make sure the FIRST line of that cell is: %%capture {cap_name} --no-display)\n")
    (Path(process_dir) / f"{cell_name}__print.txt").write_text(txt, encoding="utf-8")

def save_canvas_jpg(cell_name, process_dir):
    """
    Save canvases captured by stash_canvas() as JPG.
    """
    process_dir = Path(process_dir)
    process_dir.mkdir(parents=True, exist_ok=True)

    canvas_dict = globals().get("CELL_CANVAS_PNG", {})
    png_list = canvas_dict.get(cell_name, [])

    if not png_list:
        (process_dir / f"{cell_name}__canvas.MISSING.txt").write_text(
            f"No stashed canvas for {cell_name}. Add before plt.show(): stash_canvas('{cell_name}')\n",
            encoding="utf-8"
        )
        return

    for i, png_bytes in enumerate(png_list, 1):
        img = Image.open(io.BytesIO(png_bytes)).convert("RGB")
        img.save(process_dir / f"{cell_name}__canvas_{i:02d}.jpg", quality=95)


# -----------------------
# 0) Required metadata
# -----------------------
for k in ["TARGET_ID", "TARGET_COORDS", "SCENE_NAME"]:
    if k not in globals():
        raise RuntimeError(f"Missing {k} in globals().")

TARGET_ID = globals()["TARGET_ID"]
TARGET_COORDS = globals()["TARGET_COORDS"]
SCENE_NAME = globals()["SCENE_NAME"]

stamp = time.strftime("%Y%m%d_%H%M%S")
root = Path(f"{_safe(SCENE_NAME)}_{stamp}")

tif_dir = root / "results" / "tif"
jpg_dir = root / "results" / "jpg"
process_dir = root / "process"
meta_dir = root / "meta"
tif_dir.mkdir(parents=True, exist_ok=True)
jpg_dir.mkdir(parents=True, exist_ok=True)
process_dir.mkdir(parents=True, exist_ok=True)
meta_dir.mkdir(parents=True, exist_ok=True)

(meta_dir / "source_info.txt").write_text(
    f'TARGET_ID = "{TARGET_ID}"\nTARGET_COORDS = {TARGET_COORDS}\nSCENE_NAME = "{SCENE_NAME}"\n',
    encoding="utf-8"
)

manifest = {
    "SCENE_NAME": SCENE_NAME,
    "TARGET_ID": TARGET_ID,
    "TARGET_COORDS": TARGET_COORDS,
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "python": sys.version,
    "files": [],
}

def _add(p):
    manifest["files"].append(str(Path(p)))

def save_pair(cell_prefix, name, arr, kind):
    base = f"{cell_prefix}__{name}"
    tif_path = tif_dir / f"{base}.tif"
    jpg_path = jpg_dir / f"{base}.jpg"
    save_tif(tif_path, arr); _add(tif_path)
    if kind == "rgb":
        save_jpg_rgb01(jpg_path, arr); _add(jpg_path)
    else:
        # 默认 map 用 turbo，可视化；Cell5 的 errormap 会走专门的 scaling 保存
        m = np.asarray(arr)
        if m.ndim == 3:
            m = m[...,0]
        # 简单 robust 映射
        lo, hi = np.percentile(m.reshape(-1), 1), np.percentile(m.reshape(-1), 99)
        save_jpg_map_scaled(jpg_path, m, vmin=float(lo), vmax=float(hi), cmap="turbo"); _add(jpg_path)

# -----------------------
# Cell 1 outputs
# -----------------------
if "real_t1_cloudy" in globals():
    save_pair("cell1", "real_t1_cloudy", globals()["real_t1_cloudy"], "rgb")
if "real_t2_clear" in globals():
    save_pair("cell1", "real_t2_clear", globals()["real_t2_clear"], "rgb")
if "real_t3_clear" in globals():
    save_pair("cell1", "real_t3_clear", globals()["real_t3_clear"], "rgb")
if "real_t1_prob" in globals() and globals()["real_t1_prob"] is not None:
    base = "cell1__real_t1_prob"
    save_tif(tif_dir / f"{base}.tif", globals()["real_t1_prob"]); _add(tif_dir / f"{base}.tif")
    # prob 0..100
    save_jpg_map_scaled(jpg_dir / f"{base}.jpg", globals()["real_t1_prob"], vmin=0.0, vmax=100.0, cmap="inferno"); _add(jpg_dir / f"{base}.jpg")

save_print_txt("cell1", "cap_cell1", process_dir); _add(process_dir / "cell1__print.txt")
save_canvas_jpg("cell1", process_dir)

# -----------------------
# Cell 2b outputs
# -----------------------
if "Qwen_result" in globals() and globals()["Qwen_result"] is not None:
    Q = globals()["Qwen_result"]
    save_pair("cell2", "Qwen_Result", Q, "rgb")

    if "real_t1_cloudy" in globals():
        diff1 = np.abs(globals()["real_t1_cloudy"] - Q) * 5.0
        save_tif(tif_dir / "cell2__Diff_Cloudy_Result.tif", diff1); _add(tif_dir / "cell2__Diff_Cloudy_Result.tif")
        save_jpg_map_scaled(jpg_dir / "cell2__Diff_Cloudy_Result.jpg", np.clip(diff1,0,1), vmin=0.0, vmax=1.0, cmap="inferno"); _add(jpg_dir / "cell2b__Diff_Cloudy_Result.jpg")

    if "real_t2_clear" in globals():
        diff2 = np.abs(globals()["real_t2_clear"] - Q) * 5.0
        save_tif(tif_dir / "cell2__Diff_Clear_Result.tif", diff2); _add(tif_dir / "cell2__Diff_Clear_Result.tif")
        save_jpg_map_scaled(jpg_dir / "cell2__Diff_Clear_Result.jpg", np.clip(diff2,0,1), vmin=0.0, vmax=1.0, cmap="inferno"); _add(jpg_dir / "cell2b__Diff_Clear_Result.jpg")

save_print_txt("cell2", "cap_cell2", process_dir); _add(process_dir / "cell2__print.txt")
save_canvas_jpg("cell2", process_dir)

# -----------------------
# Cell 3 outputs
# -----------------------
if "Iq_adj" in globals():
    save_pair("cell3", "Qwen_result_adj", globals()["Iq_adj"], "rgb")
if "T" in globals():
    save_pair("cell3", "transmission", globals()["T"], "map")
if "A_rgb" in globals():
    base = "cell3__global_atmospheric_light"
    save_tif(tif_dir / f"{base}.tif", globals()["A_rgb"]); _add(tif_dir / f"{base}.tif")
    save_patch_from_rgb(jpg_dir / f"{base}.jpg", globals()["A_rgb"]); _add(jpg_dir / f"{base}.jpg")

save_print_txt("cell3", "cap_cell3", process_dir); _add(process_dir / "cell3__print.txt")
save_canvas_jpg("cell3", process_dir)

# -----------------------
# Cell 4 outputs
# -----------------------
dbg4 = globals().get("CELL4_DEBUG", None)
if isinstance(dbg4, dict):
    mapping = [
        ("Physical_inversion_J_phy", "J_phy", "rgb"),
        ("Cognitive_weight_T", "W_cog", "map"),
        ("Low_frequency_adjusted_J_cog_lf", "J_cog_lf", "rgb"),
        ("Observation_detail_weight_T", "W_detail", "map"),
        ("Cognitively_adjusted_J_cog", "J_cog", "rgb"),
        ("Visibility_weight_omega", "W_vis", "map"),
        ("Reference_weight_1_minus_omega", "W_ref", "map"),
        ("Aligned_temporal_reference", "J_ref", "rgb"),
        ("AllcloudRemoval", "J", "rgb"),
    ]
    for out_name, key, kind in mapping:
        if key in dbg4:
            save_pair("cell4", out_name, dbg4[key], kind)
else:
    (meta_dir / "cell4_missing_debug.txt").write_text(
        "CELL4_DEBUG not found. In Cell4, add: globals()['CELL4_DEBUG']={...} before return.",
        encoding="utf-8"
    )
    _add(meta_dir / "cell4_missing_debug.txt")

save_print_txt("cell4", "cap_cell4", process_dir); _add(process_dir / "cell4__print.txt")
save_canvas_jpg("cell4", process_dir)

# -----------------------
# Cell 5 outputs (recompute MAE + SAME scaling)
# -----------------------
if "err_vs_clear_mae" in globals() and globals()["err_vs_clear_mae"] is not None:
    base = "cell5__err_vs_clear_mae"
    save_tif(tif_dir / f"{base}.tif", globals()["err_vs_clear_mae"]); _add(tif_dir / f"{base}.tif")
    # prob 0..100
    save_jpg_map_scaled(jpg_dir / f"{base}.jpg", globals()["err_vs_clear_mae"], vmin=0.0, vmax=1.0, cmap="inferno"); _add(jpg_dir / f"{base}.jpg")

if "err_vs_clear_mae_Qwen" in globals() and globals()["err_vs_clear_mae_Qwen"] is not None:
    base = "cell5__err_vs_clear_mae_Qwen"
    save_tif(tif_dir / f"{base}.tif", globals()["err_vs_clear_mae_Qwen"]); _add(tif_dir / f"{base}.tif")
    # prob 0..100
    save_jpg_map_scaled(jpg_dir / f"{base}.jpg", globals()["err_vs_clear_mae_Qwen"], vmin=0.0, vmax=1.0, cmap="inferno"); _add(jpg_dir / f"{base}.jpg")


save_print_txt("cell5", "cap_cell5", process_dir); _add(process_dir / "cell5__print.txt")
save_canvas_jpg("cell5", process_dir)

# -----------------------
# Manifest + zip + download
# -----------------------
(meta_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = Path(str(root) + ".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in root.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(root)))

print("✅ Export folder:", root.resolve())
print("✅ ZIP:", zip_path.resolve())

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("⚠️ Not in Colab or download failed:", e)
